In [ ]:
!pip install lightgbm

# Идея: Среднее от показаний трех лучших моделей

Почему отказались: не отказались, по сути, финальное решение -- это доработка этого конвейера с другими гиперпараметрами

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import KFold
from sklearn.metrics import root_mean_squared_error, r2_score
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')
from catboost import CatBoostRegressor
from sklearn.ensemble import RandomForestRegressor, VotingRegressor
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_regression, mutual_info_regression
from lightgbm import LGBMRegressor
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

In [2]:
# Загружаем данные
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')


In [3]:
# Фиксируем физический минимум для защиты от деления на ноль
IC50_MIN = train['IC50, mM'].min()

# Таргеты
target_cols_original = ['IC50, mM', 'CC50, mM', 'SI']


In [4]:
# === ШАГ 1: РАСШИФРОВКА И FEATURE ENGINEERING (ХИМИЧЕСКИЕ ИНСАЙТЫ) ===
def advanced_chemistry_features(df):
    df = df.copy()
    
    # 1. Считаем общую сумму всех функциональных групп (сложность структуры)
    fr_cols = [col for col in df.columns if col.startswith('fr_')]
    df['total_fr_groups'] = df[fr_cols].sum(axis=1)
    
    # 2. Индекс проницаемости мембран (Масса / Полярная поверхность)
    df['TPSA_safe'] = np.clip(df['TPSA'], 0.1, None)
    df['mass_per_tpsa'] = df['MolWt'] / df['TPSA_safe']
    
    # 3. Удельная гидрофобность молекулы
    df['logp_per_wt'] = df['MolLogP'] / (df['MolWt'] + 1)
    
    # 4. Диапазон распределения электронного заряда
    df['charge_range'] = df['MaxPartialCharge'] - df['MinPartialCharge']
    
    # Убираем временную колонку страховки
    df = df.drop(columns=['TPSA_safe'])
    return df

In [5]:
# Применяем расшифровку к обоим датасетам
train_fe = advanced_chemistry_features(train)
test_fe = advanced_chemistry_features(test)

In [6]:
# Обновляем список признаков, добавляя новые сгенерированные дескрипторы
feature_cols = [col for col in test_fe.columns if col not in target_cols_original and col != 'index']

In [7]:
# Вычисляем целевой SI в логарифмах (наш победный сжатый таргет)
protected_train_ic50 = np.clip(train_fe['IC50, mM'], 1e-6, None)
train_fe['SI_log'] = np.log10(train_fe['CC50, mM'] / protected_train_ic50)

In [8]:
# Переводим в матрицы NumPy для максимальной скорости и защиты от IndexError
X_train = train_fe[feature_cols].values
X_test = test_fe[feature_cols].values

In [9]:
submission_final = pd.DataFrame({'index': test_fe['index']})
kf = KFold(n_splits=5, shuffle=True, random_state=42)

In [10]:
# Модель 1: Обучение для IC50
y_ic50 = train_fe['IC50, mM'].values
preds_ic50 = np.zeros(len(test_fe))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_va = X_train[train_idx], X_train[val_idx]
    y_tr, y_va = y_ic50[train_idx], y_ic50[val_idx]
    
    model = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.02,
        depth=6,
        loss_function='RMSE',
        l2_leaf_reg=6,
        random_seed=42,
        verbose=False
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], early_stopping_rounds=80, verbose=False)
    preds_ic50 += model.predict(X_test) / 5

In [11]:
# Модель 2: Обучение для CC50
y_cc50 = train_fe['CC50, mM'].values
preds_cc50 = np.zeros(len(test_fe))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_va = X_train[train_idx], X_train[val_idx]
    y_tr, y_va = y_cc50[train_idx], y_cc50[val_idx]
    
    model = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.02,
        depth=6,
        loss_function='RMSE',
        l2_leaf_reg=6,
        random_seed=42,
        verbose=False
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], early_stopping_rounds=80, verbose=False)
    preds_cc50 += model.predict(X_test) / 5

In [ ]:
# Модель 3: Обучение для логарифмического SI_log
y_si_log = train_fe['SI_log'].values
preds_si_log = np.zeros(len(test_fe))

for fold, (train_idx, val_idx) in enumerate(kf.split(X_train)):
    X_tr, X_va = X_train[train_idx], X_train[val_idx]
    y_tr, y_va = y_si_log[train_idx], y_si_log[val_idx]
    
    model = CatBoostRegressor(
        iterations=1500,
        learning_rate=0.02,
        depth=6,
        loss_function='RMSE',
        l2_leaf_reg=6,
        random_seed=42,
        verbose=False
    )
    model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], early_stopping_rounds=80, verbose=False)
    preds_si_log += model.predict(X_test) / 5

In [ ]:
# === ШАГ 4: СБОРКА И ВОЗВРАТ ИЗ ЛОГАРИФМОВ ===
submission_final['IC50'] = np.clip(preds_ic50, IC50_MIN, None)
submission_final['CC50'] = np.clip(preds_cc50, 0.0, None)

In [ ]:
# Возвращаем SI из логарифмического масштаба (10 в степени)
submission_final['SI'] = 10 ** preds_si_log
submission_final['SI'] = np.clip(submission_final['SI'], train['SI'].min(), train['SI'].max())

# Колонки
submission_final = submission_final[['index', 'IC50', 'CC50', 'SI']]

print("\nПроверка финальных средних значений:")
print(submission_final[['IC50', 'CC50', 'SI']].mean())

In [ ]:
# Сохранение финального файла
submission_final.to_csv('final_expert_chemistry_submission.csv', index=False)
print("\n[УСПЕХ] Файл 'final_expert_chemistry_submission.csv' успешно создан.")